In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
df = pd.read_csv("garments_worker_productivity.csv")
print(df.shape)
print(df.head())
print(df.isna().sum())

(1197, 15)
       date   quarter  department       day  team  targeted_productivity  \
0  1/1/2015  Quarter1      sweing  Thursday     8                   0.80   
1  1/1/2015  Quarter1  finishing   Thursday     1                   0.75   
2  1/1/2015  Quarter1      sweing  Thursday    11                   0.80   
3  1/1/2015  Quarter1      sweing  Thursday    12                   0.80   
4  1/1/2015  Quarter1      sweing  Thursday     6                   0.80   

     smv     wip  over_time  incentive  idle_time  idle_men  \
0  26.16  1108.0       7080         98        0.0         0   
1   3.94     NaN        960          0        0.0         0   
2  11.41   968.0       3660         50        0.0         0   
3  11.41   968.0       3660         50        0.0         0   
4  25.90  1170.0       1920         50        0.0         0   

   no_of_style_change  no_of_workers  actual_productivity  
0                   0           59.0             0.940725  
1                   0            

In [3]:
df["department"] = df["department"].str.strip()
df["department"] = df["department"].replace("sweing", "sewing")

df = df.drop(columns=["date"])

df["MeetsTarget"] = (df["actual_productivity"] >= df["targeted_productivity"]).astype(int)

print(df["department"].unique())
print(df["MeetsTarget"].value_counts())

['sewing' 'finishing']
MeetsTarget
1    875
0    322
Name: count, dtype: int64


In [4]:
train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index)

train_df = train_df.sort_index()
test_df = test_df.sort_index()

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
print("train index head:", train_df.index[:10].tolist())
print("test index head:", test_df.index[:10].tolist())

train shape: (958, 15)
test shape: (239, 15)
train index head: [0, 2, 3, 4, 5, 6, 7, 9, 10, 11]
test index head: [1, 8, 13, 14, 20, 21, 34, 36, 40, 53]


In [5]:
cat_cols = ["quarter", "department", "day", "team"]
num_cols = ["targeted_productivity", "smv", "wip", "over_time",
            "incentive", "idle_time", "idle_men",
            "no_of_style_change", "no_of_workers"]


def build_preprocessor():
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ])

In [6]:
X_train_reg = train_df[num_cols + cat_cols]
y_train_reg = train_df["actual_productivity"].values

X_test_reg = test_df[num_cols + cat_cols]
y_test_reg = test_df["actual_productivity"].values

pre_reg = build_preprocessor()
X_train_reg_p = pre_reg.fit_transform(X_train_reg)
X_test_reg_p = pre_reg.transform(X_test_reg)

print("processed train shape:", X_train_reg_p.shape)
print("processed test shape:", X_test_reg_p.shape)

lin = LinearRegression()

t0 = time.perf_counter()
lin.fit(X_train_reg_p, y_train_reg)
train_time_reg = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_reg = lin.predict(X_test_reg_p)
pred_time_reg = time.perf_counter() - t0

mae = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)

print()
print("Linear Regression")
print("train time:   {:.6f} s".format(train_time_reg))
print("predict time: {:.6f} s".format(pred_time_reg))
print("MAE:  {:.6f}".format(mae))
print("RMSE: {:.6f}".format(rmse))
print("R2:   {:.6f}".format(r2))

processed train shape: (958, 34)
processed test shape: (239, 34)

Linear Regression
train time:   0.003864 s
predict time: 0.001017 s
MAE:  0.101798
RMSE: 0.141897
R2:   0.348388


In [7]:
X_train_clf = train_df[num_cols + cat_cols]
y_train_clf = train_df["MeetsTarget"].values

X_test_clf = test_df[num_cols + cat_cols]
y_test_clf = test_df["MeetsTarget"].values

pre_clf = build_preprocessor()
X_train_clf_p = pre_clf.fit_transform(X_train_clf)
X_test_clf_p = pre_clf.transform(X_test_clf)

log = LogisticRegression(max_iter=1000)

t0 = time.perf_counter()
log.fit(X_train_clf_p, y_train_clf)
train_time_clf = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_clf = log.predict(X_test_clf_p)
pred_time_clf = time.perf_counter() - t0

acc = accuracy_score(y_test_clf, y_pred_clf)
prec = precision_score(y_test_clf, y_pred_clf, zero_division=0)
rec = recall_score(y_test_clf, y_pred_clf, zero_division=0)
f1 = f1_score(y_test_clf, y_pred_clf, zero_division=0)

print("Logistic Regression")
print("train time:   {:.6f} s".format(train_time_clf))
print("predict time: {:.6f} s".format(pred_time_clf))
print("accuracy:  {:.6f}".format(acc))
print("precision: {:.6f}".format(prec))
print("recall:    {:.6f}".format(rec))
print("f1:        {:.6f}".format(f1))

Logistic Regression
train time:   0.009557 s
predict time: 0.000140 s
accuracy:  0.761506
precision: 0.787879
recall:    0.912281
f1:        0.845528
